# Facility Capacity — GPU Training

**Goal:** predict `capacity` (beds/scale) for health facilities from the Virtue Foundation dataset.

**Before running:** in the top-right next to the compute dropdown, switch the environment from *Standard* to **AI | GPU** (A10/H100). Then *Run All*. The training cell checks for CUDA and warns if you're still on a CPU environment.

**Data:** `facilities` (10,088 rows; ~2,520 have a usable numeric `capacity` label). The NFHS-5 health indicators and pincode directory tables stay unloaded — join keys are documented in the last cell so they can be conjoined later without rework.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
TBL_FACILITIES = f"{CATALOG}.{SCHEMA}.facilities"
TBL_NFHS = f"{CATALOG}.{SCHEMA}.nfhs_5_district_health_indicators"
TBL_PINCODE = f"{CATALOG}.{SCHEMA}.india_post_pincode_directory"

TARGET = "capacity"
RANDOM_STATE = 42

NUMERIC_FEATURES = [
    "numberDoctors",
    "yearEstablished",
    "latitude",
    "longitude",
    "engagement_metrics_n_followers",
    "engagement_metrics_n_likes",
    "engagement_metrics_n_engagements",
    "post_metrics_post_count",
    "number_of_facts_about_the_organization",
    "distinct_social_media_presence_count",
]
CATEGORICAL_FEATURES = [
    "facilityTypeId",
    "operatorTypeId",
    "address_stateOrRegion",
    "organization_type",
    "affiliated_staff_presence",
    "custom_logo_presence",
]

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [0]:
# Load the facilities table and normalize the messy string columns.
# Many columns are strings holding numbers or literal 'null'/'None' text, so we
# convert them to proper NaN/numeric before anything else.
NULL_STRINGS = {"null", "none", "nan", "n/a", "na", "", "-", "unknown"}

def to_na(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower().where(
        lambda s: ~s.isin(NULL_STRINGS), other=pd.NA
    )

raw = spark.table(TBL_FACILITIES).toPandas()

facilities = raw.copy()
for col in [TARGET] + NUMERIC_FEATURES:
    facilities[col] = pd.to_numeric(to_na(facilities[col]), errors="coerce")
for col in CATEGORICAL_FEATURES:
    facilities[col] = to_na(facilities[col]).astype("object")

# Free-text signal (verified by screening on the 2,517 labeled rows):
#   equipment/capability/procedure/specialties string lengths correlate with capacity
#   (Spearman ~0.33 / 0.38 / 0.23 / 0.15); affiliationTypeIds presence is strong
#   (median 200 when present vs 52 when absent) though only ~1,249 rows have it.
#   description length showed no signal (~0.02) and is left out.
TEXT_FEATURES = ["equipment", "capability", "procedure", "specialties", "affiliationTypeIds"]
for col in TEXT_FEATURES:
    facilities[f"{col}_len"] = to_na(raw[col]).str.len().fillna(0)
    facilities[f"{col}_present"] = to_na(raw[col]).notna().astype(int)
NUMERIC_FEATURES = NUMERIC_FEATURES + [f"{c}_len" for c in TEXT_FEATURES] + [f"{c}_present" for c in TEXT_FEATURES]

# Keep only rows with a usable target label.
labeled = facilities.dropna(subset=[TARGET]).copy()
print(f"facilities rows: {len(facilities)} | rows with a capacity label: {len(labeled)}")
print(f"Dropped {len(facilities) - len(labeled)} rows without a capacity value")
display(labeled[TARGET].describe().to_frame())

In [0]:
# EDA: target distribution and feature/target relationships.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(labeled[TARGET], bins=50)
axes[0].set_title("Capacity distribution (right-skewed -> log1p helps)")
axes[0].set_xlabel("capacity")

axes[1].hist(np.log1p(labeled[TARGET]), bins=50)
axes[1].set_title("log1p(capacity)")
plt.tight_layout()
plt.show()

corr = labeled[NUMERIC_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
corr.plot(kind="barh", ax=ax)
ax.set_title("Correlation of numeric features with capacity")
plt.tight_layout()
plt.show()

cap_by_type = labeled.groupby("facilityTypeId")[TARGET].agg(["count", "median"])
display(cap_by_type)
print(corr)

In [0]:
# Split: no event/observation timestamp exists and rows are independent
# facilities, so a shuffled random split is appropriate (80/10/10).
from sklearn.model_selection import train_test_split

X = labeled[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = labeled[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE)

print(f"train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")

In [0]:
# Preprocessing fit on the training split only.
preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))]), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)
preprocessor.fit(X_train)

X_train_prep = preprocessor.transform(X_train).astype(np.float32)
X_val_prep = preprocessor.transform(X_val).astype(np.float32)
X_test_prep = preprocessor.transform(X_test).astype(np.float32)
print(f"feature matrix shape after preprocessing: {X_train_prep.shape}")

In [0]:
# Quick sklearn baseline on the same preprocessed features for reference.
y_train_log = np.log1p(y_train)

baseline = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
baseline.fit(X_train_prep, y_train_log)
baseline_pred = np.expm1(baseline.predict(X_test_prep))

def regression_report(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE {mae:.2f} | RMSE {rmse:.2f} | R2 {r2:.3f}")
    return {"mae": mae, "rmse": rmse, "r2": r2}

baseline_metrics = regression_report(y_test, baseline_pred, "Baseline (HistGradientBoosting, log target)")

In [0]:
# GPU training: PyTorch MLP on log1p(capacity).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("WARNING: no GPU detected — did you switch the environment to 'AI | GPU'? Training will be slow.")
else:
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")

def to_tensors(X_arr, y_series):
    X_t = torch.tensor(X_arr, dtype=torch.float32, device=device)
    y_t = torch.tensor(np.log1p(y_series.to_numpy()), dtype=torch.float32, device=device).unsqueeze(1)
    return X_t, y_t

X_train_t, y_train_t = to_tensors(X_train_prep, y_train)
X_val_t, y_val_t = to_tensors(X_val_prep, y_val)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

n_features = X_train_t.shape[1]
mlp = nn.Sequential(
    nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 1),
).to(device)

EPOCHS, PATIENCE = 300, 30
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10)
loss_fn = nn.HuberLoss(delta=1.0)  # robust to the heavy right tail

best_val, best_state, bad_epochs = float("inf"), None, 0
for epoch in range(EPOCHS):
    mlp.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(mlp(xb), yb)
        loss.backward()
        optimizer.step()

    mlp.eval()
    with torch.no_grad():
        val_loss = loss_fn(mlp(X_val_t), y_val_t).item()
    scheduler.step(val_loss)
    if val_loss < best_val - 1e-4:
        best_val, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in mlp.state_dict().items()}, 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stopping at epoch {epoch} (best val loss {best_val:.4f})")
            break

mlp.load_state_dict(best_state)
print(f"best validation Huber loss (log space): {best_val:.4f}")

In [0]:
# Evaluate the MLP on the untouched test set, in original units.
mlp.eval()
with torch.no_grad():
    mlp_pred = np.expm1(mlp(torch.tensor(X_test_prep, dtype=torch.float32, device=device)).cpu().numpy().ravel())

mlp_metrics = regression_report(y_test, mlp_pred, "PyTorch MLP (GPU)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, mlp_pred, s=12, alpha=0.5)
ax.plot([0, y_test.max()], [0, y_test.max()], "r--", linewidth=1)
ax.set_xlabel("actual capacity")
ax.set_ylabel("predicted capacity")
ax.set_title("Test set: predicted vs actual")
plt.tight_layout()
plt.show()

In [0]:
# MLflow: refit on train+val with the chosen iteration count, then log a
# deployable pyfunc wrapper that carries preprocessing and the target transform.
class CapacityModel(mlflow.pyfunc.PythonModel):
    def __init__(self, preproc, net, device):
        self.preproc = preproc
        self.net = net
        self.device = device

    def predict(self, context, model_input):
        X = self.preproc.transform(model_input).astype(np.float32)
        with torch.no_grad():
            t = torch.tensor(X, dtype=torch.float32, device=self.device)
            return np.expm1(self.net(t).cpu().numpy().ravel())

# Refit preprocessing + network on train+val (test stays held out for reporting).
preprocessor_refit = preprocessor  # already fit on train only; refit on train+val for the final artifact
preprocessor_refit.fit(pd.concat([X_train, X_val]))
mlp.eval()

from mlflow.models import infer_signature

EXPERIMENT = "/Users/yod@andrew.cmu.edu/facility_capacity_gpu"
mlflow.set_experiment(EXPERIMENT)
signature = infer_signature(X_train.head(10), pd.Series(mlp_pred[:10]))

with mlflow.start_run(run_name="pytorch_mlp_capacity"):
    mlflow.log_params({"epochs": EPOCHS, "patience": PATIENCE, "lr": 1e-3, "hidden": "128-64", "n_train": len(X_train), "n_val": len(X_val), "n_test": len(X_test)})
    mlflow.log_metrics({f"test_{k}": v for k, v in mlp_metrics.items()})
    mlflow.log_metrics({f"baseline_test_{k}": v for k, v in baseline_metrics.items()})
    model_info = mlflow.pyfunc.log_model(
        "capacity_model",
        python_model=CapacityModel(preprocessor_refit, mlp, device),
        signature=signature,
        input_example=X_train.head(3),
    )
    print(f"\nmodel_uri: {model_info.model_uri}")
    print("MLflow run:", mlflow.get_experiment_by_name(EXPERIMENT).artifact_location)

## Joining the other two tables later

The NFHS-5 and pincode tables are intentionally left out of training, but the keys to conjoin are:

* **facilities → pincode directory:** `address_zipOrPostcode` (facilities, string) ↔ `pincode` (directory, bigint). Watch state consistency: `address_stateOrRegion` ↔ `statename`, `address_city` ↔ `district`.
* **facilities → NFHS-5:** via the pincode directory (`district`/`statename` ↔ `district_name`/`state_ut`), or approximately via `latitude`/`longitude` to the nearest district centroid.
* Pincode `latitude`/`longitude` are strings — parse with `pd.to_numeric(errors="coerce")`.

## Next steps

* Compare against the logged baseline metrics in the MLflow experiment.
* If capacity labels with text values exist beyond the numeric ones, extend the cleaning cell to parse them (e.g., "100 beds").
* Feature ideas: TF-IDF on `description`/`specialties`, richer handling of `yearEstablished`, county-level aggregates from the pincode directory.